# El techo de la categoría y la redistribución de shares

## La hipótesis

Que la demanda agregada de una categoría sea un **techo** relativamente estable y
predecible, y que los productos individuales se repartan ese techo cambiando de
share — con los lanzamientos sacándole mercado a los que ya estaban, más que
agrandando la torta.

Es la lógica de un portfolio: el índice se mueve por su cuenta, los pesos se
redistribuyen entre los componentes, y la suma de pesos es 1 por construcción. Si
los shares son **mean-reverting** alrededor de un equilibrio lento, la estructura es
la de un sistema **cointegrado**: las series individuales derivan, pero su
combinación no.

En forecasting esto tiene nombre — *hierarchical / top-down forecasting* — y en
consumo masivo es la descomposición clásica **volumen de categoría × market share**:

$$ tn_{p,t} = Total_{c,t} \times share_{p,t} $$

## Lo que hay que probar para que la idea se sostenga

| # | Afirmación | Cómo se testea | Si falla |
|---|---|---|---|
| **H1** | El agregado es más predecible que las partes | WAPE de un pronóstico naive a cada nivel de agregación | no hay ganancia en pronosticar el total |
| **H2** | Los shares son mean-reverting | ADF sobre shares vs sobre niveles, y vida media del AR(1) | los shares derivan: no hay equilibrio |
| **H3** | Los lanzamientos son suma cero | *diff-in-diff* del total de la categoría al entrar un producto | el techo no existe, la torta crece |
| **H4** | Hay productos núcleo estables | share alto + baja volatilidad de share + vida larga | no hay estructura núcleo/satélite |

Y al final, lo único que decide de verdad: **un backtest top-down contra
bottom-up**. Si la descomposición no baja el WAPE, la teoría es linda y no sirve.

> Viene de `01_EDA_series.ipynb`, que ya dejó un indicio a favor de H3: los
> incumbentes caen 8,8 puntos cuando entra un producto nuevo a su `cat3`.

## 0 — Ambiente

In [ ]:
import os, json, warnings
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        return Path(env).expanduser().resolve()
    for cand in ("/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "datasets_fe"
DIR_OUT.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})

def limpiar(ax, titulo=None, y=None, x=None):
    if titulo: ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y: ax.set_ylabel(y)
    if x: ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax

HORIZONTE = 2          # el mismo que usa el pipe
print(f"BUCKET: {BUCKET}")

In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t").unique(subset=["product_id"])

def a_m(c): return (pl.col(c) // 100) * 12 + (pl.col(c) % 100)
def m_a_periodo(m): return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1

base = (sell.group_by(["product_id", "periodo"]).agg(pl.col("tn").sum().alias("tn"))
            .with_columns(a_m("periodo").alias("m")))

vida = base.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"), pl.col("m").max().alias("m_muere"),
    pl.col("tn").sum().alias("tn_total"))

grilla = (vida.select("product_id","m_nace","m_muere")
              .with_columns(pl.int_ranges("m_nace", pl.col("m_muere")+1).alias("m"))
              .explode("m").drop("m_nace","m_muere"))

panel = (grilla.join(base, on=["product_id","m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0))
               .join(prod.select("product_id","cat1","cat2","cat3","brand"), on="product_id", how="left")
               .join(vida, on="product_id", how="left")
               .sort(["product_id","m"]))

M_MIN, M_MAX = panel["m"].min(), panel["m"].max()
MESES = list(range(M_MIN, M_MAX + 1))

# Share dentro de cat3 y total de la categoria
cat_tot = panel.group_by(["cat3","m"]).agg(pl.col("tn").sum().alias("tn_cat3"))
panel = (panel.join(cat_tot, on=["cat3","m"], how="left")
              .with_columns(pl.when(pl.col("tn_cat3") > 0)
                              .then(pl.col("tn")/pl.col("tn_cat3"))
                              .otherwise(0.0).alias("share")))

mercado = panel.group_by("m").agg(pl.col("tn").sum().alias("tn_mercado")).sort("m")
print(f"panel: {panel.height:,} filas · {panel['product_id'].n_unique()} productos · "
      f"{panel['cat3'].n_unique()} cat3")
print(f"rango: {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}")

## H1 — ¿El agregado es más predecible que las partes?

La prueba más directa y la más honesta: aplicar **el mismo pronóstico naive** a cada
nivel de agregación y medir su WAPE. El naive de horizonte 2 es
`predicción(t+2) = valor(t)` — sin parámetros que ajustar, así que la única
diferencia entre niveles es cuánto ruido tiene la serie.

Si el WAPE baja al agregar, hay **ganancia por agregación**: el ruido idiosincrático
de cada producto se cancela y queda la señal común.

In [ ]:
def wape_naive(df, claves, h=HORIZONTE):
    """WAPE de predecir valor(t+h) con valor(t), sobre las series de `claves`."""
    d = df.sort(claves + ["m"])
    d = d.with_columns(pl.col("tn").shift(-h).over(claves).alias("real_fut"))
    d = d.drop_nulls("real_fut")
    err = (d["real_fut"] - d["tn"]).abs().sum()
    den = d["real_fut"].abs().sum()
    return float(err / den) if den else float("nan"), d.height


niveles = []
# producto (el nivel al que se predice de verdad)
w, n = wape_naive(panel, ["product_id"]);                    niveles.append(("producto", w, n, panel["product_id"].n_unique()))
for col, nombre in [("cat3","cat3"), ("cat2","cat2"), ("cat1","cat1"), ("brand","marca")]:
    agg = panel.group_by([col,"m"]).agg(pl.col("tn").sum()).rename({col:"k"})
    w, n = wape_naive(agg, ["k"]);                           niveles.append((nombre, w, n, agg["k"].n_unique()))
agg = panel.group_by("m").agg(pl.col("tn").sum()).with_columns(pl.lit(1).alias("k"))
w, n = wape_naive(agg, ["k"]);                               niveles.append(("mercado total", w, n, 1))

res = pl.DataFrame([{"nivel": a, "wape_naive": round(b,4), "obs": c, "series": d}
                    for a,b,c,d in niveles])
print(res)

fig, ax = plt.subplots(figsize=(7.5, 3.4))
xs = list(range(len(niveles)))
vals = [b for _,b,_,_ in niveles]
ax.bar(xs, vals, color=SERIE[0], width=.62)
for x, v in zip(xs, vals):
    ax.annotate(f"{v:.3f}", (x, v), xytext=(0,3), textcoords="offset points",
                ha="center", color=TINTA2, fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels([a for a,_,_,_ in niveles])
limpiar(ax, f"WAPE de un pronóstico naive a horizonte {HORIZONTE}, por nivel de agregación",
        y="WAPE")
plt.tight_layout(); plt.show()

g = 100*(niveles[0][1] - niveles[-1][1]) / niveles[0][1]
print(f"\nGanancia por agregacion (producto -> mercado): {g:.0f}% menos error.")
print(f"VEREDICTO H1: {'SE SOSTIENE' if g > 20 else 'NO SE SOSTIENE'}"
      f"  (umbral arbitrario: 20% de mejora)")
if g > 20:
    print("  El agregado es sustancialmente mas predecible. Ojo con la lectura:")
    print("  esto NO implica que convenga pronosticar arriba y repartir. Implica que")
    print("  las features de nivel categoria traen senial. El reparto hay que testearlo")
    print("  aparte, y es lo que hace el backtest del final.")

## H2 — ¿Los shares son mean-reverting?

El corazón de la analogía con cointegración. Se compara, **para los mismos
productos**, la estacionariedad de dos series:

- el **nivel** `tn` — se espera que derive (no estacionaria)
- el **share** dentro de `cat3` — si la hipótesis vale, debería revertir a la media

Dos medidas:

- **ADF** (Dickey-Fuller aumentado): `p < 0,05` rechaza raíz unitaria → estacionaria.
- **Vida media** del AR(1): ajustando `x_t - μ = ρ·(x_{t-1} - μ)`, la vida media es
  `ln(0,5)/ln(ρ)`. Es cuántos meses tarda en recorrer la mitad del camino de vuelta
  al equilibrio. Una vida media de 3 meses es reversión fuerte; de 24, la serie
  prácticamente deriva.

In [ ]:
MIN_OBS = 24
elegibles = (vida.filter((pl.col("m_muere") - pl.col("m_nace") + 1) >= MIN_OBS)
                 .sort("tn_total", descending=True)["product_id"].to_list())
print(f"{len(elegibles)} productos con >= {MIN_OBS} meses")

def adf_p(x):
    x = np.asarray(x, float)
    if len(x) < 12 or np.nanstd(x) == 0:
        return np.nan
    try:
        return float(adfuller(x, autolag="AIC")[1])
    except Exception:
        return np.nan

def rho_ar1(x):
    """Coeficiente AR(1) sobre la serie demeaned."""
    x = np.asarray(x, float)
    if len(x) < 12 or np.nanstd(x) == 0:
        return np.nan
    d = x - x.mean()
    a, b = d[:-1], d[1:]
    if (a**2).sum() == 0:
        return np.nan
    return float((a*b).sum() / (a**2).sum())


def vida_media(x):
    """Half-life del AR(1). NaN si la serie no revierte de forma suave:
    rho>=1 deriva (raiz unitaria), rho<=0 oscila de signo.
    OJO: esos NaN se excluyen de la mediana, asi que la mediana de las que SI
    revierten no alcanza para concluir nada. Por eso tambien se reporta rho."""
    rho = rho_ar1(x)
    if rho is None or not np.isfinite(rho) or not (0 < rho < 1):
        return np.nan
    return float(np.log(.5) / np.log(rho))

filas = []
for pid, g in panel.filter(pl.col("product_id").is_in(elegibles)).sort(["product_id","m"]).group_by("product_id", maintain_order=True):
    pid = pid[0] if isinstance(pid, tuple) else pid
    filas.append({"product_id": pid,
                  "adf_p_nivel": adf_p(g["tn"].to_numpy()),
                  "adf_p_share": adf_p(g["share"].to_numpy()),
                  "rho_nivel": rho_ar1(g["tn"].to_numpy()),
                  "rho_share": rho_ar1(g["share"].to_numpy()),
                  "hl_nivel": vida_media(g["tn"].to_numpy()),
                  "hl_share": vida_media(g["share"].to_numpy())})
est = pl.DataFrame(filas)

pn = est["adf_p_nivel"].drop_nulls(); ps = est["adf_p_share"].drop_nulls()
print(f"\nestacionarias (ADF p<0.05)")
print(f"   nivel tn : {100*(pn<.05).mean():5.1f}%   (n={pn.len()})")
print(f"   share    : {100*(ps<.05).mean():5.1f}%   (n={ps.len()})")
rn = est["rho_nivel"].drop_nulls(); rs = est["rho_share"].drop_nulls()
print(f"\ncoeficiente AR(1) (mediana)")
print(f"   nivel tn : {rn.median():+.3f}")
print(f"   share    : {rs.median():+.3f}")
print(f"\nvida media de reversion (mediana, meses) -- SOLO sobre las que revierten")
print(f"   nivel tn : {est['hl_nivel'].median():.1f}   "
      f"(revierten {100*est['hl_nivel'].drop_nulls().len()/est.height:.0f}% de las series)")
print(f"   share    : {est['hl_share'].median():.1f}   "
      f"(revierten {100*est['hl_share'].drop_nulls().len()/est.height:.0f}% de las series)")
print()
print("Una vida media de ~1 mes NO es 'reversion fuerte a un equilibrio': significa")
print("rho bajo, o sea que la serie es basicamente ruido alrededor de su media.")
print("La cointegracion que buscamos necesitaria rho alto (0.7-0.95) con reversion")
print("LENTA hacia un equilibrio movil. Con rho ~0.5 no hay equilibrio que seguir.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

ax = axes[0]
bins = np.linspace(0, 1, 26)
ax.hist(pn.to_numpy(), bins=bins, color=MUDO, alpha=.6, label="nivel tn")
ax.hist(ps.to_numpy(), bins=bins, histtype="step", color=SERIE[0], linewidth=2, label="share")
ax.axvline(.05, color=SERIE[1], linewidth=1.5)
ax.annotate("p=0,05", (.05, ax.get_ylim()[1]*.92), xytext=(6,0), textcoords="offset points",
            color=SERIE[1], fontsize=8)
limpiar(ax, "Test ADF: p-valor", y="productos", x="p-valor (menor = más estacionaria)")
ax.legend(fontsize=8)

ax = axes[1]
hn = est["hl_nivel"].drop_nulls().to_numpy(); hs = est["hl_share"].drop_nulls().to_numpy()
bins = np.linspace(0, 24, 25)
ax.hist(np.clip(hn, 0, 24), bins=bins, color=MUDO, alpha=.6, label="nivel tn")
ax.hist(np.clip(hs, 0, 24), bins=bins, histtype="step", color=SERIE[0], linewidth=2, label="share")
limpiar(ax, "Vida media de reversión a la media", y="productos", x="meses (recortado en 24)")
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
# Veredicto derivado de los datos, no escrito a mano.
mas_estacionario = (ps < .05).mean() > (pn < .05).mean()
rho_alto = rs.median() >= 0.7
print("VEREDICTO H2")
print(f"  share mas estacionario que el nivel : {'SI' if mas_estacionario else 'NO'}"
      f"  ({100*(ps<.05).mean():.0f}% vs {100*(pn<.05).mean():.0f}%)")
print(f"  rho del share alto (>=0.7)          : {'SI' if rho_alto else 'NO'}"
      f"  (mediana {rs.median():+.3f})")
if mas_estacionario and rho_alto:
    print("\n  -> H2 SE SOSTIENE: hay un equilibrio lento al que el share vuelve.")
elif mas_estacionario:
    print("\n  -> H2 SE SOSTIENE A MEDIAS. El share es mas estacionario que el nivel,")
    print("     pero con rho bajo eso significa 'ruido alrededor de una media fija',")
    print("     no 'reversion lenta a un equilibrio movil'. No es la estructura")
    print("     cointegrada que justificaria un modelo top-down: es simplemente que")
    print("     el share de un producto no se mueve mucho de un mes al otro.")
else:
    print("\n  -> H2 NO SE SOSTIENE: el share no es mas estable que el nivel.")

## H3 — ¿Los lanzamientos son suma cero? (el techo)

El test decisivo de tu idea. Cuando entra un producto nuevo a una `cat3`:

- Si la categoría **no crece**, el entrante se comió share de los demás → **hay techo**.
- Si la categoría crece justo el volumen del entrante → la torta se agrandó, **no hay techo**.

El problema es que entre "antes" y "después" pasan cosas ajenas al lanzamiento —
estacionalidad, tendencia del mercado. Por eso no alcanza con comparar la categoría
consigo misma: se usa **diferencias en diferencias**, tomando el resto del mercado
como grupo de control.

```
Δ esperada  = total_cat3(antes) × (mercado(después) / mercado(antes) − 1)
Δ atribuible = total_cat3(después) − total_cat3(antes) − Δ esperada
ratio       = Δ atribuible / volumen del entrante
```

`ratio ≈ 0` → canibalización pura (techo). `ratio ≈ 1` → expansión pura (sin techo).

In [ ]:
V = 6      # ventana a cada lado
lanz = (vida.filter((pl.col("m_nace") > M_MIN + V) & (pl.col("m_nace") <= M_MAX - V))
            .join(prod.select("product_id","cat3"), on="product_id", how="left"))

merc = dict(zip(mercado["m"].to_list(), mercado["tn_mercado"].to_list()))
cat_d = {}
for c3, m, t in zip(cat_tot["cat3"], cat_tot["m"], cat_tot["tn_cat3"]):
    cat_d[(c3, m)] = t
tn_d = {}
for p, m, t in zip(panel["product_id"], panel["m"], panel["tn"]):
    tn_d[(p, m)] = t

filas = []
for pid, m0, c3 in zip(lanz["product_id"], lanz["m_nace"], lanz["cat3"]):
    pre  = range(m0 - V, m0)
    post = range(m0 + 1, m0 + V + 1)
    cat_pre  = sum(cat_d.get((c3, m), 0.0) for m in pre)
    cat_post = sum(cat_d.get((c3, m), 0.0) for m in post)
    mer_pre  = sum(merc.get(m, 0.0) for m in pre)
    mer_post = sum(merc.get(m, 0.0) for m in post)
    v_new    = sum(tn_d.get((pid, m), 0.0) for m in post)
    if cat_pre <= 0 or mer_pre <= 0 or v_new <= 0:
        continue
    esperada   = cat_pre * (mer_post / mer_pre - 1.0)
    atribuible = (cat_post - cat_pre) - esperada
    filas.append({"product_id": pid, "cat3": c3, "vol_entrante": v_new,
                  "delta_atribuible": atribuible,
                  "ratio_expansion": atribuible / v_new})

exp = pl.DataFrame(filas)
print(f"{exp.height} lanzamientos evaluados\n")
r = exp["ratio_expansion"].to_numpy()
print(f"ratio de expansion   mediana: {np.median(r):+.2f}   media: {r.mean():+.2f}")
print(f"   canibalizacion pura o mas (ratio <= 0)  : {100*(r<=0).mean():.0f}%")
print(f"   parcial            (0 < ratio < 1)      : {100*((r>0)&(r<1)).mean():.0f}%")
print(f"   expansion pura o mas (ratio >= 1)       : {100*(r>=1).mean():.0f}%")

# Ponderado por volumen: los lanzamientos grandes pesan mas
w = exp["vol_entrante"].to_numpy()
r_pond = float((r*w).sum()/w.sum())
print(f"\nratio ponderado por volumen del entrante: {r_pond:+.2f}")
print("\nVEREDICTO H3")
if np.median(r) < 0.5 and r_pond < 0.5:
    print("  SE SOSTIENE: los lanzamientos son mayormente redistribucion. Hay techo.")
elif np.median(r) > 1 or r_pond > 1:
    print("  NO SE SOSTIENE: la categoria crece MAS que el volumen del entrante.")
    print("  La torta se agranda; no hay techo fijo que repartir.")
    print("  Cuidado: esto convive con que los incumbentes SI pierdan volumen.")
    print("  Canibalizacion y expansion no son excluyentes: el entrante puede")
    print("  quitarle ventas a los que estaban Y traer demanda nueva al mismo tiempo.")
    print("  Y recorda el sesgo de endogeneidad: se lanza donde ya venia creciendo.")
else:
    print("  MIXTO: parte redistribucion, parte expansion.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(-2, 3, 51)
ax.hist(np.clip(r, -2, 3), bins=bins, color=SERIE[0], alpha=.75)
ax.axvline(0, color=SERIE[1], linewidth=2)
ax.axvline(1, color=SERIE[2], linewidth=2)
ax.axvline(np.median(r), color=TINTA, linewidth=1.5, linestyle=(0,(4,3)))
top = ax.get_ylim()[1]
ax.annotate("0 = canibalización pura\n(hay techo)", (0, top*.95), xytext=(-8,0),
            textcoords="offset points", ha="right", va="top", color=SERIE[1], fontsize=8)
ax.annotate("1 = expansión pura\n(no hay techo)", (1, top*.95), xytext=(8,0),
            textcoords="offset points", va="top", color=SERIE[2], fontsize=8)
ax.annotate(f"mediana {np.median(r):+.2f}", (np.median(r), top*.55), xytext=(8,0),
            textcoords="offset points", color=TINTA, fontsize=8)
limpiar(ax, f"¿Cuánto de un lanzamiento es torta nueva? (diff-in-diff, n={exp.height})",
        y="lanzamientos", x="ratio de expansión (recortado en [-2, 3])")
plt.tight_layout(); plt.show()

### Dos límites de este test, antes de creerle

**Los lanzamientos no son aleatorios.** Las empresas lanzan en categorías que ya
vienen creciendo, así que parte del crecimiento posterior habría ocurrido igual. El
diff-in-diff controla por el mercado entero, pero no por la tendencia propia de esa
`cat3`. Un control mejor sería emparejar cada categoría tratada con categorías de
tendencia previa parecida. Con este sesgo, el ratio de expansión está
**sobreestimado**: la canibalización real es mayor que la que muestra el histograma.

**El ratio tiene cola pesada.** El denominador es el volumen del entrante, y los
lanzamientos chicos generan ratios enormes en valor absoluto. Por eso la mediana y
el promedio ponderado por volumen dicen cosas distintas del promedio simple: mirá
esos dos y no el promedio crudo.

## H4 — Productos núcleo y satélites

Un **núcleo** es lo que en un portfolio serían las posiciones estructurales: share
alto, share **estable**, presencia continua. Los **satélites** entran y salen, y su
share oscila.

Tres criterios, todos sobre el share dentro de `cat3`:

- `share_medio` alto — pesa en su categoría
- `cv_share` bajo — su participación no se mueve mucho (coeficiente de variación)
- `cobertura` alta — está presente en casi todos los meses de la ventana

In [ ]:
perfil = (panel.filter(pl.col("product_id").is_in(elegibles))
    .group_by("product_id").agg(
        pl.col("share").mean().alias("share_medio"),
        pl.col("share").std().alias("share_sd"),
        pl.len().alias("meses"),
        pl.col("tn").sum().alias("tn_total"),
        pl.col("cat3").first().alias("cat3"))
    .with_columns(
        (pl.col("share_sd") / pl.when(pl.col("share_medio") > 0)
                               .then(pl.col("share_medio")).otherwise(1.0)).alias("cv_share"),
        (pl.col("meses") / len(MESES)).alias("cobertura")))

um_share = perfil["share_medio"].median()
um_cv    = perfil["cv_share"].median()
perfil = perfil.with_columns(
    pl.when((pl.col("share_medio") >= um_share) & (pl.col("cv_share") <= um_cv)
            & (pl.col("cobertura") >= .8)).then(pl.lit("nucleo"))
      .when((pl.col("share_medio") < um_share) & (pl.col("cv_share") > um_cv)).then(pl.lit("satelite"))
      .otherwise(pl.lit("intermedio")).alias("tipo"))

print(perfil.group_by("tipo").agg(
    pl.len().alias("productos"),
    pl.col("tn_total").sum().alias("tn"),
    pl.col("share_medio").median().round(4).alias("share_medio_mediano"),
    pl.col("cv_share").median().round(2).alias("cv_mediano"))
    .with_columns((100*pl.col("tn")/perfil["tn_total"].sum()).round(1).alias("%_tn"))
    .sort("%_tn", descending=True))

fig, ax = plt.subplots(figsize=(7, 4.4))
for j, t in enumerate(["nucleo", "satelite", "intermedio"]):
    d = perfil.filter(pl.col("tipo") == t)
    ax.scatter(d["cv_share"], d["share_medio"], s=14,
               color=[SERIE[0], SERIE[1], MUDO][j], alpha=.6, label=t)
ax.axhline(um_share, color=EJE_C, linewidth=1); ax.axvline(um_cv, color=EJE_C, linewidth=1)
ax.set_xscale("log"); ax.set_yscale("log")
limpiar(ax, "Núcleos vs satélites", y="share medio en su cat3 (log)", x="CV del share (log)")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

## El veredicto: top-down contra bottom-up

Todo lo anterior es diagnóstico. Esto es la prueba.

Se predice `tn` de cada producto a horizonte 2 de dos maneras, usando **componentes
igual de simples** en ambas — así la única diferencia es la descomposición y no la
sofisticación del modelo:

| | cómo predice `tn_p(t+2)` |
|---|---|
| **bottom-up** | media móvil de 3 meses de `tn_p` |
| **top-down** | media móvil de 3 meses de `Total_cat3` × media móvil de 3 meses de `share_p` |

Si H1 y H2 valen, el top-down debería ganar: el total es más estable (H1) y el share
revierte, así que su media móvil es un buen estimador (H2).

Se agrega también el naive puro como piso de referencia. La métrica es **WAPE a
nivel producto**, la misma del pipe.

In [ ]:
d = panel.sort(["product_id","m"]).with_columns(
    pl.col("tn").rolling_mean(3, min_periods=3).over("product_id").alias("tn_ma3"),
    pl.col("share").rolling_mean(3, min_periods=3).over("product_id").alias("share_ma3"),
)
ct = (cat_tot.sort(["cat3","m"])
             .with_columns(pl.col("tn_cat3").rolling_mean(3, min_periods=3).over("cat3").alias("cat3_ma3")))
d = d.join(ct.select("cat3","m","cat3_ma3"), on=["cat3","m"], how="left")

# real futuro a t+HORIZONTE
d = d.with_columns(pl.col("tn").shift(-HORIZONTE).over("product_id").alias("real_fut"))
d = d.drop_nulls(["real_fut","tn_ma3","share_ma3","cat3_ma3"])

d = d.with_columns(
    pl.col("tn").alias("pred_naive"),
    pl.col("tn_ma3").alias("pred_bottomup"),
    (pl.col("cat3_ma3") * pl.col("share_ma3")).alias("pred_topdown"),
)

def wape(real, pred):
    real = np.asarray(real, float); pred = np.maximum(np.asarray(pred, float), 0)
    return float(np.abs(real - pred).sum() / np.abs(real).sum())

R = d["real_fut"].to_numpy()
resultados = [
    ("naive (repetir t)",        wape(R, d["pred_naive"].to_numpy())),
    ("bottom-up (MM3 del tn)",   wape(R, d["pred_bottomup"].to_numpy())),
    ("top-down (total × share)", wape(R, d["pred_topdown"].to_numpy())),
]
for n, w in resultados:
    print(f"{n:28s} WAPE {w:.4f}")

mejor_bu = resultados[1][1]; td = resultados[2][1]
dif = 100*(mejor_bu-td)/mejor_bu
print(f"\ntop-down vs bottom-up: {dif:+.1f}%")
print(f"n = {d.height:,} observaciones producto-mes")
print("\nVEREDICTO")
if abs(dif) < 1:
    print("  EMPATE TECNICO. La descomposicion no aporta por si sola.")
    print("  No invalida la idea entera: las features de categoria y share pueden")
    print("  seguir sirviendo COMO INPUT de un GBM, que es distinto de usarlas para")
    print("  reconstruir la prediccion a mano. Eso se testea en el leaderboard.")
elif dif > 0:
    print("  GANA TOP-DOWN. Vale la pena explorar la reconstruccion jerarquica.")
else:
    print("  GANA BOTTOM-UP. Predecir directo al nivel del producto es mejor que")
    print("  pronosticar el total y repartirlo.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
nombres = [n for n,_ in resultados]; vals = [w for _,w in resultados]
cols = [MUDO, SERIE[1], SERIE[0]]
b = ax.barh(range(len(vals)), vals, color=cols, height=.6)
for i, v in enumerate(vals):
    ax.annotate(f"{v:.4f}", (v, i), xytext=(6,0), textcoords="offset points",
                va="center", color=TINTA2, fontsize=9)
ax.set_yticks(range(len(vals))); ax.set_yticklabels(nombres)
ax.invert_yaxis(); ax.set_xlim(0, max(vals)*1.18)
limpiar(ax, f"WAPE a nivel producto, horizonte {HORIZONTE}", x="WAPE")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); plt.show()

In [ ]:
# ¿Donde gana cada uno? Por tipo de producto y por fase de vida.
d2 = d.join(perfil.select("product_id","tipo"), on="product_id", how="left")
d2 = d2.with_columns(pl.col("tipo").fill_null("sin_perfil"))

filas = []
for t, g in d2.group_by("tipo"):
    t = t[0] if isinstance(t, tuple) else t
    r = g["real_fut"].to_numpy()
    if len(r) < 100 or np.abs(r).sum() == 0:
        continue
    filas.append({"tipo": t, "n": g.height,
                  "wape_bottomup": round(wape(r, g["pred_bottomup"].to_numpy()), 4),
                  "wape_topdown":  round(wape(r, g["pred_topdown"].to_numpy()), 4)})
comp = (pl.DataFrame(filas)
          .with_columns((100*(pl.col("wape_bottomup")-pl.col("wape_topdown"))
                         / pl.col("wape_bottomup")).round(1).alias("ganancia_topdown_%"))
          .sort("ganancia_topdown_%", descending=True))
print(comp)
print("\nUna ganancia concentrada en un tipo de producto es mas util que una pareja:")
print("permite usar top-down solo donde sirve y bottom-up en el resto (modelo hibrido).")

## Features exportables

Todas calculadas con información disponible **hasta el mes de la fila** (medias
móviles y acumulados hacia atrás, nada de máximos globales ni fechas de muerte).
Se pegan por `(product_id, periodo)`.

In [ ]:
feats = (d.select(
        "product_id", "m", "cat3",
        pl.col("share").alias("share_cat3"),
        pl.col("share_ma3").alias("share_cat3_ma3"),
        (pl.col("share") - pl.col("share_ma3")).alias("share_desvio_vs_ma3"),
        pl.col("tn_cat3"),
        pl.col("cat3_ma3").alias("tn_cat3_ma3"),
        pl.col("pred_topdown").alias("pred_topdown_h2"),
    )
    .with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo"))
    .drop("m")
    .join(perfil.select("product_id","tipo","share_medio","cv_share","cobertura"),
          on="product_id", how="left")
    .join(est.select("product_id","hl_share","adf_p_share"), on="product_id", how="left"))

out = DIR_OUT / "features_share_categoria.parquet"
feats.write_parquet(out)
print(f"Guardado: {out}")
print(f"{feats.height:,} filas x {feats.width} columnas\n")
print(feats.columns)

### Advertencia sobre dos de estas columnas

`tipo`, `share_medio`, `cv_share`, `hl_share` y `adf_p_share` se calculan sobre
**toda** la serie del producto, así que miran el futuro. Para explorar y segmentar
están bien; para entrenar hay que recalcularlas con un corte, igual que los clusters
de DTW: sólo con meses anteriores al último mes de train.

Las que sí son seguras tal cual: `share_cat3`, `share_cat3_ma3`,
`share_desvio_vs_ma3`, `tn_cat3`, `tn_cat3_ma3` y `pred_topdown_h2`.

`pred_topdown_h2` es interesante como feature: es la predicción top-down metida
**como input** del modelo, en vez de como alternativa. Así el GBM decide cuánto
hacerle caso, producto por producto — que suele funcionar mejor que elegir uno de
los dos enfoques a mano.

## Qué probar en el pipe

1. Pegar `share_cat3_ma3`, `share_desvio_vs_ma3` y `tn_cat3_ma3` al dataset de
   `02_FE` y correr `03_Optuna` con `sufijo='conShare'`. Comparar en el leaderboard.
2. Pegar `pred_topdown_h2` como feature y ver si el modelo la usa (mirá su lugar en
   `importancia.csv`).
3. Si la tabla por tipo de producto mostró ganancia concentrada en los núcleos,
   probar un modelo híbrido: top-down para esos, el pipe normal para el resto.